In [1]:
# In[1]:


# =============================================================================
# CONFIG + CONSTANTS
# =============================================================================
from pyspark.sql import functions as F
from delta.tables import DeltaTable

GOLD_SCHEMA = "lh_jde_gold.rpt"

ENV             = "dev"
TRIGGER         = {"processingTime": "30 seconds"}
CKPT            = f"Files/checkpoints/eso5_dim_uss_plant_{ENV}"   # OWN root — independent of other notebooks
OVERWRITE       = True    # ⚠ ONE-OFF full reprocess — set back to False after a healthy run

SRC_SCHEMA    = "jde_cdc"     # CDF must be enabled on F0005.
SRC_LAKEHOUSE = "lh_jde_silver"
F0005_TBL     = "f0005_user_defined_code_values"
UDC_SYS, UDC_TYPE = "55", "UP"   # ✅ CONFIRMED from the core query (DRSY='55' AND DRRT='UP') — see header

T_DIM  = f"{GOLD_SCHEMA}.dim_uss_plant"

print(f"ESO5 Gold dim_uss_plant processor — trigger {TRIGGER}  target {T_DIM}")

StatementMeta(, 95f51282-62de-41f1-b32c-80e1fcee25eb, 3, Finished, Available, Finished, False)

ESO5 Gold dim_uss_plant processor — trigger {'processingTime': '30 seconds'}  target lh_jde_gold.rpt.dim_uss_plant


In [2]:
# In[2]:


# =============================================================================
# HELPERS  (identical to the ESO4 dim notebooks)
# =============================================================================

_SOFT_DELETE_COLS = ["is_delete", "deleted_date_time"]

def sname(table_name):
    return f"{SRC_LAKEHOUSE}.{SRC_SCHEMA}.{table_name}"

def load_silver_table(table_name):
    df = spark.table(sname(table_name))
    if "is_delete" in df.columns:
        df = df.filter(F.col("is_delete") == 0)
    return df.select(*[c for c in df.columns if c not in _SOFT_DELETE_COLS])

def current_version(silver_table):
    return spark.sql(f"DESCRIBE HISTORY {sname(silver_table)}").select(F.max("version")).first()[0]

StatementMeta(, 95f51282-62de-41f1-b32c-80e1fcee25eb, 4, Finished, Available, Finished, False)

In [3]:
# In[3]:


# =============================================================================
# DIM transform — dim_uss_plant. Natural PK = the UDC value (DRKY) WITHIN its system/type, so the
# transform filters product_code/user_defined_codes FIRST, then keys on trim(user_defined_code) cast
# to the numeric vendor (Hubble: TO_NUMBER(rtrim(F0005.drky)) = M.sdvend). `restrict_keys` (carrying
# vendor_number) narrows the CDC recompute.  [shape: eso4/nb/nb_eso4_gold_dim_udc.py::_udc_dim]
#
# ⚠ KEY TYPE = DOUBLE, NOT long. Direct Lake requires the dim PK and the fact FK to have the SAME
# physical type. The fact's `loading_facility` is F4211 `primary_last_vendor_no` — a JDE numeric that
# lands as Double — and the reused rpt.dim_address_book keys on `address_number` (Double) for the same
# reason. Casting DRKY to long here yields Int64 and Fabric rejects the relationship:
#   "data types of Direct Lake relationship between FK 'fact...'[loading_facility](Double) and
#    PK 'dim_uss_plant'[vendor_number](Int64) are incompatible".
# =============================================================================
DIM_KEY      = "vendor_number"
DIM_KEY_TYPE = "double"          # MUST match fact.loading_facility (Double) — see note above

def transform_dim_uss_plant(restrict_keys=None):
    f0005 = (load_silver_table(F0005_TBL)
             .where((F.trim(F.col("product_code")) == UDC_SYS) &
                    (F.trim(F.col("user_defined_codes")) == UDC_TYPE)))
    if restrict_keys is not None:
        f0005 = f0005.join(restrict_keys.alias("r"),
                           F.trim(f0005["user_defined_code"]).cast(DIM_KEY_TYPE) == F.col(f"r.{DIM_KEY}"),
                           "left_semi")
    sphd = F.trim(F.col("special_handling_code")).cast("double")
    return (f0005.select(
                F.trim(F.col("user_defined_code")).cast(DIM_KEY_TYPE).alias(DIM_KEY),     # DRKY (numeric)
                F.when((sphd > 1) & (sphd < 9000), F.lit("Y")).otherwise(F.lit("N")).alias("uss_plant_sand"),
                F.when(sphd > 9000, F.lit("TRANSLOAD"))
                 .when((sphd > 1) & (sphd < 9000), F.lit("PLANT"))
                 .otherwise(F.lit("3RDPARTY")).alias("shipped_from"),
                F.trim(F.col("special_handling_code")).alias("lofa_mcu"))                 # LOFAPLANTMCU (raw DRSPHD)
            .where(F.col(DIM_KEY).isNotNull())
            .dropDuplicates([DIM_KEY]))

StatementMeta(, 95f51282-62de-41f1-b32c-80e1fcee25eb, 5, Finished, Available, Finished, False)

In [4]:
# In[4]:


# =============================================================================
# CDC WRITE HELPERS — NO audit columns
#   • dim : MERGE upsert (whenMatchedUpdateAll / whenNotMatchedInsertAll) + MERGE delete.
# =============================================================================
def _write_new_table(df, target, cdf=True):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
    w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if cdf:
        w = w.option("delta.enableChangeDataFeed", "true")
    w.saveAsTable(target)

def _upsert_dim(target, key_col, transform, change_keys, delete_keys):
    """MERGE upsert changed codes + MERGE delete removed codes. `*_keys` carry `key_col`."""
    n = 0
    if not change_keys.rdd.isEmpty():
        src = transform(restrict_keys=change_keys)
        (DeltaTable.forName(spark, target).alias("t")
            .merge(src.alias("s"), f"t.{key_col} = s.{key_col}")
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        n = src.count()
    if not delete_keys.rdd.isEmpty():
        (DeltaTable.forName(spark, target).alias("t")
            .merge(delete_keys.alias("s"), f"t.{key_col} = s.{key_col}")
            .whenMatchedDelete().execute())
    return n

StatementMeta(, 95f51282-62de-41f1-b32c-80e1fcee25eb, 6, Finished, Available, Finished, False)

In [5]:
# In[5]:


# =============================================================================
# FULL LOAD vs RESUME  (streaming approach from ESO4 dim notebooks)
# =============================================================================
_CKPT_PATHS = [f"{CKPT}/dim__{F0005_TBL}"]

def _checkpoints_exist():
    """True iff the per-stream checkpoint has a COMMITTED offset (offsets/ non-empty), so an
    incomplete checkpoint forces a FULL LOAD rather than cold-starting the CDF reader at
    startingVersion=0 (v0 predates CDF enablement -> DELTA_MISSING_CHANGE_DATA)."""
    for p in _CKPT_PATHS:
        try:
            if not mssparkutils.fs.ls(f"{p}/offsets"):
                return False
        except Exception:
            return False
    return True

_STREAM_NAMES = {"dim__" + F0005_TBL}
_stopped = []
for _q in list(spark.streams.active):
    if _q.name in _STREAM_NAMES:
        _q.stop()
        _stopped.append(_q.name)
if _stopped:
    print(f"Stopped leftover streams: {_stopped}")

_FULL_LOAD = OVERWRITE or not spark.catalog.tableExists(T_DIM) or not _checkpoints_exist()

if _FULL_LOAD:
    print("== FULL LOAD ==")
    spark.sql(f"DROP TABLE IF EXISTS {T_DIM}")
    _write_new_table(transform_dim_uss_plant(), T_DIM)
    print(f"  ✓ seeded {T_DIM}")
    _init_ver = {F0005_TBL: current_version(F0005_TBL)}
    print(f"  init versions: {_init_ver}")
    try:
        mssparkutils.fs.rm(CKPT, True)
        print("  checkpoints cleared")
    except Exception as e:
        print(f"  checkpoint clear skipped: {e}")
    print("✓ full load complete")
else:
    print("== RESUME from checkpoint ==")
    _init_ver = {}

StatementMeta(, 95f51282-62de-41f1-b32c-80e1fcee25eb, 7, Finished, Available, Finished, True)

== FULL LOAD ==
  ✓ seeded lh_jde_gold.rpt.dim_uss_plant
  init versions: {'f0005_user_defined_code_values': 16}
  checkpoints cleared
✓ full load complete


In [6]:
# In[6]:


# =============================================================================
# STREAM BATCH HANDLER  (structure from ESO4's nb_eso4_gold_dim_udc.py)
#   The F0005 stream carries EVERY UDC — split out the 55/UP rows, then upsert the dim.
# =============================================================================
def make_dim_uss_plant_handler(init_ver):
    def handler(batch_df, batch_id):
        if batch_df.rdd.isEmpty():
            return
        if init_ver >= 0:
            batch_df = batch_df.filter(F.col("_commit_version") > init_ver)
        if batch_df.rdd.isEmpty():
            return

        def _keys(sys_code, type_code, key_alias):
            sel = batch_df.where((F.trim(F.col("product_code")) == sys_code) &
                                 (F.trim(F.col("user_defined_codes")) == type_code))
            up = (sel.filter(F.col("_change_type").isin("insert", "update_postimage"))
                     .select(F.trim(F.col("user_defined_code")).cast(DIM_KEY_TYPE).alias(key_alias))
                     .where(F.col(key_alias).isNotNull()).distinct())
            dele = (sel.filter(F.col("_change_type") == "delete")
                       .select(F.trim(F.col("user_defined_code")).cast(DIM_KEY_TYPE).alias(key_alias))
                       .where(F.col(key_alias).isNotNull()).distinct())
            return up, dele

        up, dele = _keys(UDC_SYS, UDC_TYPE, DIM_KEY)
        n = _upsert_dim(T_DIM, DIM_KEY, transform_dim_uss_plant, up, dele)
        print(f"[{F0005_TBL[:12]}] dim_uss_plant batch={batch_id} upserts={n}")
    return handler

StatementMeta(, 95f51282-62de-41f1-b32c-80e1fcee25eb, 8, Finished, Available, Finished, True)

In [7]:
# In[7]:


# =============================================================================
# START STREAM — Silver Change Data Feed -> foreachBatch -> CDC write, every 30 s.
# REQUIRES delta.enableChangeDataFeed = true on the source (F0005).
# =============================================================================

def _start_ver(iv, tbl):
    """Full load: init_ver (exists, carries CDF; handler skips <= it). Resume (iv < 0): fall back
    to the source's CURRENT version, never 0 (v0 predates CDF)."""
    return iv if iv >= 0 else current_version(tbl)

iv_dim = _init_ver.get(F0005_TBL, -1)
_sv_dim = _start_ver(iv_dim, F0005_TBL)
(spark.readStream.format("delta")
     .option("readChangeFeed",  "true")
     .option("startingVersion", _sv_dim)
     .table(sname(F0005_TBL))
 .writeStream
     .foreachBatch(make_dim_uss_plant_handler(iv_dim))
     .option("checkpointLocation", f"{CKPT}/dim__{F0005_TBL}")
     .trigger(**TRIGGER)
     .queryName("dim__" + F0005_TBL)
     .start())
print(f"  dim__{F0005_TBL}  startingVersion={_sv_dim}  init_ver={iv_dim}")

print(f"== started 1 stream — continuous, trigger {TRIGGER}. Target {T_DIM}. ==")
spark.streams.awaitAnyTermination()

StatementMeta(, 95f51282-62de-41f1-b32c-80e1fcee25eb, 9, Finished, Available, Finished, True)

  dim__f0005_user_defined_code_values  startingVersion=16  init_ver=16
== started 1 stream — continuous, trigger {'processingTime': '30 seconds'}. Target lh_jde_gold.rpt.dim_uss_plant. ==


StreamingQueryException: [STREAM_FAILED] Query [id = 383a3ecb-0647-433d-8afb-7d826686bae5, runId = faa34407-e3fd-45a5-bd8e-da7e2ed675ed] terminated with exception: [DELTA_MISSING_CHANGE_DATA] Error getting change data for range [16 , 17] as change data was not
recorded for version [16]. If you've enabled change data feed on this table,
use `DESCRIBE HISTORY` to see when it was first enabled.
Otherwise, to start recording change data, use `ALTER TABLE table_name SET TBLPROPERTIES
(delta.enableChangeDataFeed=true)`.